opening data

In [2]:
import glob

lst = glob.glob("data/subset*/subset*/*.mhd")
lst

['data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.105756658031515062000744821260.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.108197895896446896160048741492.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524522225658609808059.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.111172165674661221381920536987.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.122763913896761494371822656720.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.124154461048929153767743874565.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.126121460017257137098781143514.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.126264578931778258890371755354.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.128023902651233986592378348912.mhd',
 'data\\subset0\\subset0\\1.3.6.1.4.1.14519.5.2.1.6279.6001.129055977637338639741695800950.mhd',
 'data\\subset0\\subset0\\1.3.

In [2]:
# a named tuple that will hold all the informatiuion

from collections import namedtuple

candidateInfoTuple = namedtuple(
    "candidateInfoTuple",
    "isNodule_bool , diameter_mm , series_uid ,center_xyz",
)

In [8]:
import glob
import os
import functools

#this will get this names of all the files in subset 0 and 1 and drop the .mhd

present_on_dist_set = ()

@functools.lru_cache(1)
def getCandidateInfoList(requireOnDisk_bool=True):
    mhd_lst = glob.glob("data/subset*/subset*/*.mhd")
    present_on_dist_set = {os.path.split(p)[-1][:-4] for p in mhd_lst}

getCandidateInfoList()

In [ ]:
import csv

"""
returns a dictonary of this format
{
abc : [(123,123),(456,753)],
gbh : [(147,4562),(45,4211)]
}
"""

diameter_dict = {}
with open("data/annotations.csv" , "r") as f:
    for row in list(csv.reader(f))[1:]:
        series_uid = row[0]
        annotationCenter_xyz = tuple(float(x) for x in row[1:4])
        annotationDiameter_mm = float(row[4])

        diameter_dict.setdefault(series_uid, [] ).append(
            (annotationCenter_xyz , annotationDiameter_mm)
        )

In [ ]:
candidateInfo_list = []

with open("data/candidates.csv","r") as f:
    for row in list(csv.reader(f))[1:]:
        series_uid = row[0]
        if series_uid not in present_on_dist_set:
            continue

        isNodule_bool = bool(int(row[4]))
        candidateCenter_xyz = tuple([float(x) for x in row[1:4]])

        candidateDiameter_mm = 0.0
        for annotation_tup in diameter_dict.get(series_uid,[]):
            annotationCenter_xyz , annotationDiameter_mm = annotation_tup
            for i in range(3):
                delta_mm = abs(candidateCenter_xyz[i] - annotationCenter_xyz[i])
